# 09 - Validacion del agente: nivel N1 (herramientas y verificador)

Rama: `08AgentValidationAndSimulator`. Sigue el protocolo `docx/03Evidencia/08-11-26_ProtocoloValidacionAgente.md`, escrito antes de ejecutar. Este cuaderno **no entrena ni recalibra** nada y **no usa un LLM**: prueba las herramientas del agente y el verificador con estados reales y modificados.

| Parte | Que hace | Publica en Kaggle |
|---|---|---|
| 1 | Elige los casos con la semilla 2026 (test de A, B, C y errores muy seguros) y extrae sus observaciones de las repeticiones publicas | `agente/validacion/casos.csv`, `observaciones.json`, `partidas.json` |
| 2 | Bateria N1: escenarios normales, limite, entradas inesperadas, otras poblaciones, sinteticos y errores del modelo | `agente/validacion/resultados_n1.csv`, `detalle_n1.csv`, `latencias_n1.csv` |
| 3 | Analisis: poblaciones, errores muy seguros y direccion de los cambios | nada |
| 4 | Catalogo de cartas para el simulador: PV maximos de cada carta vista como activo en el train de A | `agente/cartas.json` |

El nivel N2 (el agente completo con Claude) corre en la PC con `10_validacion_llm`, sobre los mismos casos.

In [31]:
# PREPARACION COMPLETA DEL CUADERNO (rama 08AgentValidationAndSimulator)
# PROPOSITO: dejar el servidor listo: repo en la rama 07, librerias fijadas, tablas del
# dataset del equipo en /content/datos y los archivos del modelo final en /content/resultados.
# ESPERADO: la linea de nucleos/RAM, "Tablas listas | caracteristicas: 62" y
# "Modelo final y metricas de test listos" y el SHA-256 del modelo del agente.
import json
import subprocess
import sys
import time
from pathlib import Path

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "08AgentValidationAndSimulator"
REPO_DIR = Path("/content/repo")
if not REPO_DIR.exists():
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo fetch origin {BRANCH}
    !git -C /content/repo checkout {BRANCH}
    !git -C /content/repo pull --ff-only origin {BRANCH}
%cd /content/repo
!pip install -q -r requirements.txt
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for nombre in [m for m in list(sys.modules) if m.split(".")[0] in ("ptcg_wp", "ptcg_agente")]:
    del sys.modules[nombre]

from ptcg_wp.colab import recursos, setup_kaggle
RECURSOS = recursos()
setup_kaggle()

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn import __version__ as sklearn_version
assert sklearn_version == "1.7.2", (f"Se requiere scikit-learn 1.7.2; version cargada: {sklearn_version}. "
                                    "Reinicia el runtime y ejecuta esta celda de nuevo.")
from ptcg_wp import store
from ptcg_wp.features import feature_columns
from ptcg_wp.ingest import load_days
from ptcg_wp.predictor import FICHA_EXPORTADA, MODELO_EXPORTADO
from ptcg_agente import artefactos

DATA_DIR = store.DATOS_DIR
RESULTADOS = store.RESULTADOS_DIR
DATASET = store.load_dataset_id()
store.fetch(DATASET, DATA_DIR, store.expected_files(load_days("configs/dias.yaml")))
schema = pq.ParquetFile(sorted(DATA_DIR.glob("estados_*.parquet"))[0]).schema.names
assert len(feature_columns(schema)) == 62
print("Tablas listas | caracteristicas: 62")

BASE = [MODELO_EXPORTADO, FICHA_EXPORTADA, *artefactos.METRICAS_TEST, artefactos.VEREDICTO]
assert store.listo(BASE, RESULTADOS), "Faltan el modelo final o las metricas de test en Kaggle"
print("Modelo final y metricas de test listos")

COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
DIRTY = bool(subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip())
print("rama:", BRANCH, "| commit:", COMMIT[:7], "| cambios locales:", DIRTY)

from ptcg_agente.modelo import ModeloAgente
from ptcg_agente.registro import Registro

artefactos.preparar(RESULTADOS)
MODELO = ModeloAgente.cargar(RESULTADOS)
REGISTRO = Registro("validacion-n1", MODELO.modelo_sha256, COMMIT[:7],
                    carpeta=RESULTADOS / "agente" / "registros")
print("modelo del agente:", MODELO.ficha["modelo"], "| SHA-256:", MODELO.modelo_sha256[:16])

remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 18 (delta 13), reused 18 (delta 13), pack-reused 0 (from 0)
Unpacking objects: 100% (18/18), 32.49 KiB | 2.95 MiB/s, done.
From https://github.com/Adriel23456/Applied-AI-Agent-Project
 * branch            08AgentValidationAndSimulator -> FETCH_HEAD
   9774075..9dcb138  08AgentValidationAndSimulator -> origin/08AgentValidationAndSimulator
Already on '08AgentValidationAndSimulator'
Your branch is behind 'origin/08AgentValidationAndSimulator' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/Adriel23456/Applied-AI-Agent-Project
 * branch            08AgentValidationAndSimulator -> FETCH_HEAD
Updating 9774075..9dcb138
Fast-forward
 .../03Evidencia/08-11-26_ValidacionN1Corregida.csv |   36 +
 notebooks/09_validacion_agente.ipynb               | 2447 +++++++++++++++++++-
 notebooks/10_va

In [32]:
# PRUEBAS DEL AGENTE Y DE LA VALIDACION
# PROPOSITO: correr las pruebas automaticas del agente, de la bateria de validacion y del
# modelo exportado, con datos sinteticos y sin Kaggle.
# ESPERADO: "N passed" sin "failed" y la linea "OK: pruebas aprobadas".
resultado = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_agente.py",
                            "tests/test_validacion.py", "tests/test_predictor.py"],
                           cwd=REPO_DIR, capture_output=True, text=True)
print(resultado.stdout[-2000:], resultado.stderr[-2000:])
assert resultado.returncode == 0, "fallaron pruebas: revisa la salida"
print("OK: pruebas aprobadas")

.......................................                                  [100%]
39 passed in 5.80s
 
OK: pruebas aprobadas


In [33]:
# PARTE 1 - CASOS DE VALIDACION (agente/validacion/)
# PROPOSITO: elegir los estados del protocolo con la semilla 2026 y extraer sus observaciones de
# las repeticiones publicas de Kaggle (una descarga por partida, unas 150-200 partidas).
# Lee test de A, B y C: ya se usaron en la evaluacion final; aqui no se elige ni ajusta ningun modelo.
# ESPERADO: 90 estados normales de A, 90 de B, 90 de C, los errores muy seguros por etapa y la
# partida mas larga de test de A. Si ya esta en Kaggle, solo se baja.
# Las dos primeras lineas son para reconstruir: quitan del dataset el partidas.json.gz de la
# primera corrida (Kaggle no lo entregaba) y guardan la eleccion publicada para comprobar que
# la nueva es identica. Si no hay nada anterior, no hacen nada.
from ptcg_agente.validacion import casos as cv

cv.quitar_formato_anterior(RESULTADOS)
ANTERIOR = cv.casos_publicados(RESULTADOS)


def preparar_casos():
    inicio = time.time()
    tabla = cv.tabla_test_a(DATA_DIR, MODELO.predictor)
    print(f"estados utiles de test de A: {len(tabla):,} | partidas: {tabla['game_id'].nunique():,}")
    elegidos = cv.elegir(tabla)
    cv.comprobar_misma_eleccion(ANTERIOR, elegidos)
    larga = cv.partida_mas_larga(DATA_DIR, tabla)
    print("partida mas larga:", larga, "| partidas a bajar:", elegidos["game_id"].nunique() + 1)
    cv.extraer(elegidos, larga, RESULTADOS)
    print(f"listo en {(time.time() - inicio) / 60:.1f} min")


store.asegurar([cv.CASOS, cv.OBSERVACIONES, cv.PARTIDAS], preparar_casos,
               "agente: casos de validacion (protocolo 08-11-26)", RESULTADOS)
DATOS = cv.Datos(RESULTADOS)
display(DATOS.casos.groupby(["grupo", "stage"]).size().unstack(fill_value=0)[list(cv.ETAPAS)])
print("repeticiones completas:", len(DATOS.repeticiones), "| partida mas larga:", DATOS.larga)

formato anterior: no hay partidas.json.gz en el indice
ya local: agente/validacion/casos.csv
ya local: agente/validacion/casos.csv
ya local: agente/validacion/observaciones.json
ya local: agente/validacion/partidas.json
=> Todo estaba en Kaggle: se bajo y NO se entreno.


stage,inicio,medio,final
grupo,,,
error_A,10,10,10
error_B,10,10,10
error_C,10,10,10
normal_A,30,30,30
poblacion_B,30,30,30
poblacion_C,30,30,30


repeticiones completas: 11 | partida mas larga: 86195831


In [34]:
# PARTE 2 - BATERIA N1 (agente/validacion/resultados_n1.csv)
# PROPOSITO: correr cada caso del protocolo (secciones 3.1 a 3.6) y guardar si aprobo, fallo o
# es solo registro, el detalle por estado y la latencia de cada herramienta.
# ESPERADO: la tabla por grupo, las metricas de la seccion 5 y una fila por caso. L-10 deberia
# fallar (prediccion del protocolo); M-02 es solo registro.
# Esta es la corrida ORIGINAL: queda fija en Kaggle como evidencia y no se recalcula aunque el
# codigo cambie. La corrida con las correcciones de la rama esta en el cuaderno 10 (parte 2).
from ptcg_agente.validacion import bateria, resumen


def correr_bateria():
    resultados, detalle, latencias = bateria.correr_n1(MODELO, DATOS, REGISTRO)
    for tabla, ruta in ((resultados, bateria.RESULTADOS_N1), (detalle, bateria.DETALLE_N1),
                        (latencias, bateria.LATENCIAS_N1)):
        tabla.to_csv(RESULTADOS / ruta, index=False)


store.asegurar([bateria.RESULTADOS_N1, bateria.DETALLE_N1, bateria.LATENCIAS_N1], correr_bateria,
               "agente: bateria de validacion N1", RESULTADOS)
RES = pd.read_csv(RESULTADOS / bateria.RESULTADOS_N1)
DET = pd.read_csv(RESULTADOS / bateria.DETALLE_N1)
LAT = pd.read_csv(RESULTADOS / bateria.LATENCIAS_N1)
display(resumen.por_grupo(RES))
display(resumen.metricas(RES, LAT))
with pd.option_context("display.max_colwidth", 200):
    display(RES[["caso", "aprobado", "obtenido"]])

ya local: agente/validacion/resultados_n1.csv
ya local: agente/validacion/detalle_n1.csv
ya local: agente/validacion/latencias_n1.csv
=> Todo estaba en Kaggle: se bajo y NO se entreno.


estado,aprobado,fallido,registro
grupo,,,
entradas,9,0,0
errores,1,0,1
limite,8,2,0
normales,6,0,0
poblaciones,2,0,0
robustez,1,0,0
sinteticos,5,0,0


,metrica,valor,meta,cumple
0,Advertencias cuando corresponden,13/13 casos,100 %,True
1,Rechazos correctos,7/7 casos,100 %,True
2,Robustez,0 excepciones,0,True
3,Latencia estimar_probabilidad (n=574),"p50 41 ms, p95 72 ms",p95 < 2000 ms,True
4,Latencia evaluar_partida (n=11),"p50 6594 ms, p95 33979 ms",p95 < 30000 ms,False
5,Latencia explicar_prediccion (n=91),"p50 112 ms, p95 131 ms",p95 < 2000 ms,True


,caso,aprobado,obtenido
0,N-01,True,dif. maxima con PredictorVictoria 0.00e+00; con la tabla 1.11e-16; verificados 90/90
1,N-02,True,SHAP que no suma: 0/90
2,N-03,True,con etapa_inicio: 30/30
3,N-04,True,con etapa_inicio o sin_prediccion: 0/60
4,N-05,True,fuera_de_dominio: 0/90
5,N-06,True,10 repeticiones; dif. maxima 5.00e-05; verificadas 10/10
6,L-01,True,"p=None, advertencias=['sin_prediccion']"
7,L-02,True,"p=None, advertencias=['sin_prediccion']"
8,L-03,True,"p=None, advertencias=['sin_prediccion']"
9,L-04,True,41.5 % -> 15.8 % (baja); advertencias=[]


In [35]:
# PARTE 3 - ANALISIS
# PROPOSITO: comparar A, B y C; medir cuantos errores muy seguros llegan sin advertencia (M-02,
# fragilidad); y ver la direccion de los cambios en los casos limite L-04 a L-07.
# ESPERADO: tres tablas y la lista de casos fallidos.
DET["acierto"] = (DET["p"] >= 0.5) == DET["y"].astype(bool)
DET["con_advertencia"] = DET["advertencias"].fillna("").ne("")
DET["fuera_de_dominio"] = DET["advertencias"].fillna("").str.contains("fuera_de_dominio")

poblaciones = DET[DET["grupo"].isin(["normal_A", "poblacion_B", "poblacion_C"])]
print("Poblaciones (fraccion por etapa):")
display(poblaciones.groupby(["grupo", "etapa"])[["acierto", "con_advertencia", "fuera_de_dominio"]]
        .mean().round(3).unstack("etapa"))

errores = DET[DET["grupo"].fillna("").str.startswith("error_")]
print("Errores muy seguros: cuantos llegan con alguna advertencia (M-01 y M-02):")
display(errores.groupby(["grupo", "etapa"]).agg(casos=("caso", "size"), con_advertencia=("con_advertencia", "mean"),
                                                p_media=("p", "mean")).round(3))

print("Direccion de los cambios en los casos limite (para la validacion humana):")
with pd.option_context("display.max_colwidth", 200):
    display(RES[RES["caso"].isin(["L-04", "L-05", "L-06", "L-07"])][["caso", "descripcion", "obtenido"]])
fallidos = RES[RES["aprobado"].astype(str) == "False"]
print("Casos fallidos:", fallidos["caso"].tolist() or "ninguno")

Poblaciones (fraccion por etapa):


acierto               con_advertencia                \
etapa         final inicio  medio           final inicio  medio   
grupo                                                             
normal_A      0.967  0.467  0.733           0.267    1.0  0.000   
poblacion_B   0.800  0.600  0.833           0.167    1.0  0.100   
poblacion_C   0.833  0.500  0.800           0.100    1.0  0.067   

            fuera_de_dominio               
etapa                  final inicio medio  
grupo                                      
normal_A                 0.0    0.0   0.0  
poblacion_B              0.0    0.0   0.0  
poblacion_C              0.0    0.0   0.0

Errores muy seguros: cuantos llegan con alguna advertencia (M-01 y M-02):


casos  con_advertencia  p_media
grupo   etapa                                  
error_A final      10              0.1    0.503
        inicio     10              1.0    0.776
        medio      10              0.2    0.426
error_B final      10              0.2    0.674
        inicio     10              1.0    0.762
        medio      10              0.1    0.414
error_C final      10              0.4    0.581
        inicio     10              1.0    0.762
        medio      10              0.0    0.844

Direccion de los cambios en los casos limite (para la validacion humana):


,caso,descripcion,obtenido
9,L-04,A-final-00 con la banca propia vacia,41.5 % -> 15.8 % (baja); advertencias=[]
10,L-05,A-final-00 con el mazo propio en 0,41.5 % -> 9.6 % (baja); advertencias=[]
11,L-06,A-final-00 con el mazo del rival en 0,41.5 % -> 98.6 % (sube); advertencias=[]
12,L-07,A-final-00 con un premio por tomar del jugador,41.5 % -> 75.8 % (sube); advertencias=[]


Casos fallidos: ['L-09', 'L-10']


In [ ]:
# PARTE 4 - CATALOGO DE CARTAS PARA EL SIMULADOR (agente/cartas.json)
# PROPOSITO: los PV maximos de TODAS las cartas que aparecieron como Pokemon activo en el train
# de A, para que el simulador los muestre al elegir una carta (las repeticiones solo traen los PV
# de las cartas que estan en juego). Si ya esta en Kaggle, solo se baja.
# ESPERADO: unas 360 cartas, cada una con sus PV maximos, y las 10 mas frecuentes.
import json as _json

from ptcg_agente import cartas
from ptcg_wp.data import load_scenario


def construir_cartas():
    estados, _, particion, _ = load_scenario(DATA_DIR, "A", include_cards=True, columns=cartas.COLUMNAS)
    catalogo = cartas.construir(estados.loc[particion == "train"])
    catalogo["commit"] = COMMIT
    destino = RESULTADOS / cartas.RUTA
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(_json.dumps(catalogo, indent=1), encoding="utf-8")


store.asegurar([cartas.RUTA], construir_cartas, "agente: catalogo de cartas (PV maximos, train de A)", RESULTADOS)
CATALOGO = pd.DataFrame(_json.loads((RESULTADOS / cartas.RUTA).read_text(encoding="utf-8"))["cartas"])
print("cartas en el catalogo:", len(CATALOGO), "| PV maximos entre", CATALOGO["maxHp"].min(), "y", CATALOGO["maxHp"].max())
display(CATALOGO.sort_values("apariciones", ascending=False).head(10))